# Dokumentähnlichkeit mit Sentence Transformers

In diesem Notebook trainieren wir ein Sentence-Transformer-Modell, das semantisch ähnliche Fragen und Antworten in einem gemeinsamen Vektorraum abbildet. Als Trainingsdaten verwenden wir die deutsche Version des Natural-Questions-Datensatzes.

## Umgebung vorbereiten

Zunächst legen wir fest, welche GPU für das Training verwendet werden soll. Die Angabe `0` wählt die erste verfügbare CUDA-GPU aus.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Daten vorbereiten

Der deutsche Natural-Questions-Datensatz liegt als komprimierte JSONL-Datei vor. Die Hilfsfunktion entfernt nicht benötigte Spalten, benennt Frage und Antwort einheitlich und reserviert 1.000 Beispiele für die Auswertung.

In [ ]:
from datasets import load_dataset

def load_nq_german(data_file = "./data/ng_german.jsonl.gz"):
    # JSONL-Datei als Datensatz laden
    dataset = (
        load_dataset("json", data_files=data_file, split="train",num_proc=8)
        .remove_columns(["query", "answer"])
        .rename_column("question_de", "query")
        .rename_column("answer_de", "answer")
    )
    dataset_dict = dataset.train_test_split(test_size=1_000, seed=12)
    return dataset_dict


In [ ]:
import logging
import random

import numpy
import torch
#from torch import mps  # noqa: F401
#torch.mps.device = mps
from datasets import Dataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss, CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

In [ ]:
logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)
random.seed(12)
torch.manual_seed(12)
numpy.random.seed(12)

## Modell konfigurieren

Wir verwenden ein mehrsprachiges Sentence-Transformer-Modell als Ausgangspunkt. Die Zufallswerte aus der vorherigen Zelle machen die Experimente besser reproduzierbar. Über die beiden Schalter lässt sich steuern, ob Rollen-Prompts verwendet und beim Pooling berücksichtigt werden.

In [ ]:
# Diese Variablen können angepasst werden:
use_prompts = True
include_prompts_in_pooling = True

# 1. Zu trainierendes Modell und optionale Model-Card-Daten festlegen
base_model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

In [ ]:
model = SentenceTransformer(
    base_model_name,
    #tokenizer_kwargs={"max_seq_length": 512},
    model_card_data=SentenceTransformerModelCardData(
        language="de",
        license="apache-2.0",
        model_name=f"{base_model_name} trained on german Natural Questions pairs",
    ),
).to(torch.bfloat16)

In [ ]:
model.set_pooling_include_prompt(include_prompts_in_pooling)

## Prompts definieren

Die Präfixe kennzeichnen, ob ein Text eine Suchanfrage oder ein Dokument ist. Dadurch kann das Modell beide Rollen beim Erzeugen der Embeddings unterscheiden.

In [ ]:
# 2. Optional: Prompts definieren
if use_prompts:
    query_prompt = "query: "
    corpus_prompt = "document: "
    prompts = {
        "query": query_prompt,
        "answer": corpus_prompt,
    }

## Trainingsdaten erzeugen

Nun laden wir den Datensatz mit der zuvor definierten Hilfsfunktion und teilen ihn in Trainings- und Evaluierungsdaten auf.

In [ ]:
# 3. Datensatz für das Training laden
dataset_dict = load_nq_german()
train_dataset: Dataset = dataset_dict["train"]
eval_dataset: Dataset = dataset_dict["test"]


## Verlustfunktion auswählen

`CachedMultipleNegativesRankingLoss` vergleicht passende Frage-Antwort-Paare mit den übrigen Beispielen eines Batches. Das Caching erlaubt dabei eine größere effektive Batch-Größe bei begrenztem GPU-Speicher.

In [ ]:
# 4. Verlustfunktion definieren
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=16) 
#loss = MultipleNegativesRankingLoss(model) # <- funktioniert auch mit MPS (Apple Silicon)


## Training konfigurieren

Für das Training legen wir unter anderem die Anzahl der Epochen, Batch-Größen, Lernrate und Auswertungsintervalle fest. Eine Viertel-Epoche verkürzt hier die Laufzeit während der Entwicklung.

In [ ]:
# 5. Optionale Trainingsparameter festlegen

# TODO: Größe des Trainingsdatensatzes für Tests begrenzen

run_name = "nq-german-" + base_model_name.split("/")[-1]
if use_prompts:
    run_name += "-prompts"
if not include_prompts_in_pooling:
    run_name += "-exclude-pooling-prompts"
args = SentenceTransformerTrainingArguments(
    # Erforderlicher Parameter:
    output_dir=f"models/{run_name}",
    # Optionale Trainingsparameter:
    num_train_epochs=0.25, # Training während der Entwicklung auf eine Viertel-Epoche begrenzen
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    learning_rate=4e-5,
    warmup_ratio=0.1,   
    fp16=False,  # Auf False setzen, falls die GPU kein FP16 unterstützt
    bf16=True,  # Auf True setzen, falls die GPU BF16 unterstützt
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # Die Verlustfunktion profitiert von Batches ohne Duplikate
    # Optionale Parameter für Protokollierung und Debugging:
    eval_strategy="steps",
    eval_steps=0.5,
    save_strategy="steps",
    save_steps=0.5,
    save_total_limit=2,
    logging_steps=5,
    logging_first_step=True,
    run_name=run_name,  # Wird von W&B verwendet, falls `wandb` installiert ist
    seed=12,
    prompts=prompts if use_prompts else None,
)

## Das Training \o/

Jetzt verbinden wir Modell, Trainingsparameter, Datensätze und Verlustfunktion in einem Trainer. Während des Trainings wird das Modell regelmäßig auf den Evaluierungsdaten geprüft.

In [ ]:
# 6. Trainer erstellen und Training starten
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    #evaluator=dev_evaluator,
)
trainer.train()

In [ ]:
# 7. Trainiertes Modell speichern
model.save_pretrained(f"models/{run_name}/final")

## Modell testen

Wir laden das gespeicherte Modell, erzeugen Embeddings für einige Beispielsätze und berechnen deren paarweise Ähnlichkeit. Hohe Werte zeigen an, dass zwei Texte im Vektorraum nahe beieinanderliegen.

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Vortrainiertes Sentence-Transformer-Modell laden
model = SentenceTransformer(f"./models/{run_name}/final")

# Zu kodierende Sätze
sentences = [
    "query: Das ist eine Frage über Kekse.",
    "answer: Das hier ist ein Keksrezept",
    "query: Ich mag Möven, oder?",
    "answer: Ich bin Paul die Möve",
]

# 2. Embeddings mit model.encode() berechnen
embeddings = model.encode(sentences)


similarities = model.similarity(embeddings, embeddings)
print(similarities)

## Embeddings visualisieren

Für den [TensorFlow Embedding Projector](https://projector.tensorflow.org/) speichern wir die Embeddings und ihre Beschriftungen in zwei tabulatorgetrennten Dateien. Dort lassen sich die Vektoren interaktiv in zwei oder drei Dimensionen untersuchen.

In [ ]:
import numpy as np
# Dateien für den TensorFlow Embedding Projector konvertieren

# Als TSV speichern
np.savetxt('output.tsv', embeddings, delimiter='\t', fmt='%g')

with open("description.tsv","w") as f:
    f.writelines(sentence + "\n" for sentence in sentences)

# Tutorial:

Das Modell kann nun semantisch ähnliche Fragen und Antworten erkennen. Machen Sie sich mit dem Notebook vertraut und untersuchen Sie, wie sich verschiedene Einstellungen auf die Ähnlichkeitswerte auswirken.

**Ihre Aufgabe ist es, die Qualität der Dokumentähnlichkeit zu verbessern.**

Hier sind einige Ideen:

* Vergleichen Sie das Training mit und ohne Rollen-Prompts. Prüfen Sie auch, welchen Einfluss `include_prompts_in_pooling` hat.

* Testen Sie andere mehrsprachige oder deutschsprachige Modelle aus dem [Model Hub](https://huggingface.co/models?library=sentence-transformers).

* Variieren Sie Lernrate, Batch-Größe und Anzahl der Epochen. Bewerten Sie Änderungen nicht nur mit einzelnen Beispielen, sondern mit einem geeigneten Evaluierungsdatensatz.

* Ergänzen Sie eigene Frage-Antwort-Paare und betrachten Sie deren Position im TensorFlow Embedding Projector.